# 18C2 — Frozen SHARP Independent Cycle-25 Evaluation

## Purpose

Evaluate the already-frozen SHARP-only Cycle-24 pipelines on the independent **2021–2025 Cycle-25 test set**.

This notebook must not fit or modify:

- base-model parameters;
- preprocessing parameters;
- calibration mappings;
- decision thresholds;
- feature definitions;
- sample-selection rules.

## Frozen pipeline source

The Cycle-24 pipeline was frozen in 18B.

For each model:

1. load the saved base estimator;
2. generate raw probabilities for Cycle 25;
3. apply the saved Platt calibrator;
4. apply the saved Cycle-24 threshold;
5. report independent-test metrics.

## Independent test set

Input tensor produced by 18C1:

- years: 2021–2025;
- shape expected: `(N, 3, 15)`;
- 15 SHARP features at t−288, t−192, t−96 minutes;
- conservative QUALITY=0 rule;
- no Cycle-25 tuning.

## Metrics

Overall and by year:

- ROC-AUC
- PR-AUC
- Brier score
- log loss
- TSS
- HSS
- precision
- recall
- F1
- TN / FP / FN / TP

## Scientific rule

These are the first independent cross-cycle SHARP results.

No threshold or calibrator may be altered after viewing Cycle-25 performance in this notebook.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

HOME = Path.home()

TEST = HOME / "aia_sharp_cycle25_input"
FREEZE = HOME / "sharp_cycle24_freeze_20260917"
OUT = HOME / "sharp_cycle25_evaluation_20260917"

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(exist_ok=True)

EPS = 1e-6

print("TEST:", TEST)
print("FREEZE:", FREEZE)
print("OUT:", OUT)


## 1. Load and lock the independent Cycle-25 test set

In [ ]:
X3 = np.load(TEST / "sharp_cycle25_test_X.npy", mmap_mode="r")
y = np.load(TEST / "sharp_cycle25_test_y.npy", mmap_mode="r")
rows = pd.read_csv(TEST / "sharp_cycle25_test_rows.csv.gz")

protocol_18c1 = json.loads((TEST / "protocol_record.json").read_text())

assert protocol_18c1["status"] == "CYCLE25_SHARP_TEST_ARRAYS_BUILT_NO_MODEL_EVALUATION"
assert protocol_18c1["cycle25_used_for_tuning"] is False
assert X3.shape == (45433, 3, 15)
assert y.shape == (45433,)
assert len(rows) == 45433
assert sorted(rows["stored_year"].unique().tolist()) == [2021, 2022, 2023, 2024, 2025]
assert np.isfinite(X3).all()

X45 = np.asarray(X3).reshape(len(X3), 45)

print("X3:", X3.shape)
print("X45:", X45.shape)
print("positives:", int(y.sum()))
print("regions:", rows["region_component_id"].nunique())
print(rows["stored_year"].value_counts().sort_index().to_string())


## 2. Load the frozen 18B models, calibrators and thresholds

In [ ]:
freeze_summary = json.loads((FREEZE / "cycle24_freeze_summary.json").read_text())
freeze_protocol = json.loads((FREEZE / "protocol_record.json").read_text())

assert freeze_protocol["status"] == "CYCLE24_REFERENCE_PIPELINE_FROZEN_PENDING_INDEPENDENT_CYCLE25_EVALUATION"
assert freeze_protocol["cycle25_used"] is False

frozen = {}

for rec in freeze_summary:
    model_name = rec["model"]

    base_model = joblib.load(
        FREEZE / "models" / f"{model_name}__base_model.joblib"
    )
    calibrator = joblib.load(
        FREEZE / "models" / f"{model_name}__platt_calibrator.joblib"
    )
    threshold = float(rec["selected_threshold"])

    frozen[model_name] = {
        "model": base_model,
        "calibrator": calibrator,
        "threshold": threshold,
    }

    print(model_name, "frozen threshold =", threshold)

assert set(frozen) == {"logistic_regression", "random_forest"}


## 3. Metric helpers — no threshold search

In [ ]:
def tss_from_cm(tn, fp, fn, tp):
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    return tpr - fpr

def hss_from_cm(tn, fp, fn, tp):
    num = 2 * (tp * tn - fn * fp)
    den = (tp + fn) * (fn + tn) + (tp + fp) * (fp + tn)
    return num / den if den else np.nan

def logit(p):
    p = np.clip(np.asarray(p), EPS, 1-EPS)
    return np.log(p / (1-p)).reshape(-1, 1)

def apply_platt(calibrator, raw_p):
    return calibrator.predict_proba(logit(raw_p))[:, 1]

def evaluate(y_true, p, threshold):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    # Ranking metrics require both classes.
    roc = float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else np.nan
    pr = float(average_precision_score(y_true, p)) if len(np.unique(y_true)) == 2 else np.nan

    return {
        "n": int(len(y_true)),
        "positives": int(np.sum(y_true)),
        "prevalence": float(np.mean(y_true)),
        "roc_auc": roc,
        "pr_auc": pr,
        "brier": float(brier_score_loss(y_true, p)),
        "log_loss": float(log_loss(y_true, np.clip(p, EPS, 1-EPS), labels=[0, 1])),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tss": float(tss_from_cm(tn, fp, fn, tp)),
        "hss": float(hss_from_cm(tn, fp, fn, tp)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }


## 4. Run the frozen independent Cycle-25 evaluation exactly once

In [ ]:
overall_records = []
year_records = []

for model_name, bundle in frozen.items():
    model = bundle["model"]
    calibrator = bundle["calibrator"]
    threshold = bundle["threshold"]

    raw_p = model.predict_proba(X45)[:, 1]
    calibrated_p = apply_platt(calibrator, raw_p)

    overall = evaluate(y, calibrated_p, threshold)
    overall_records.append({
        "model": model_name,
        **overall,
    })

    pred = rows[
        ["target_sample_id", "HARPNUM", "NOAA_AR_clean", "region_component_id", "stored_year"]
    ].copy()
    pred["y_true"] = y
    pred["raw_probability"] = raw_p
    pred["calibrated_probability"] = calibrated_p
    pred["frozen_threshold"] = threshold
    pred["prediction"] = (calibrated_p >= threshold).astype(int)

    pred.to_csv(
        OUT / "predictions" / f"{model_name}__cycle25_2021_2025.csv.gz",
        index=False,
        compression="gzip",
    )

    for year in [2021, 2022, 2023, 2024, 2025]:
        mask = rows["stored_year"].to_numpy() == year
        rec = evaluate(y[mask], calibrated_p[mask], threshold)
        year_records.append({
            "model": model_name,
            "year": year,
            **rec,
        })

overall_df = pd.DataFrame(overall_records)
year_df = pd.DataFrame(year_records)

print("===== OVERALL 2021–2025 =====")
print(overall_df.to_string(index=False))

print("\n===== BY YEAR =====")
print(year_df.to_string(index=False))


## 5. Save immutable evaluation evidence

In [ ]:
overall_df.to_csv(OUT / "cycle25_overall_results.csv", index=False)
year_df.to_csv(OUT / "cycle25_yearly_results.csv", index=False)

with open(OUT / "cycle25_overall_results.json", "w") as f:
    json.dump(overall_records, f, indent=2)

with open(OUT / "cycle25_yearly_results.json", "w") as f:
    json.dump(year_records, f, indent=2)

protocol = {
    "status": "FROZEN_SHARP_PIPELINES_EVALUATED_ON_INDEPENDENT_CYCLE25_2021_2025",
    "test_years": [2021, 2022, 2023, 2024, 2025],
    "test_shape": list(X3.shape),
    "test_positives": int(y.sum()),
    "test_regions": int(rows["region_component_id"].nunique()),
    "models": list(frozen.keys()),
    "base_model_refit_on_cycle25": False,
    "preprocessing_refit_on_cycle25": False,
    "calibration_refit_on_cycle25": False,
    "threshold_reselected_on_cycle25": False,
    "feature_selection_on_cycle25": False,
    "sample_selection_changed_after_results": False,
    "scientific_clearance": False,
    "limitations": [
        "Earlier label-clearance and SHARP historical-availability limitations remain documented.",
        "This notebook reports the independent SHARP cross-cycle evaluation only.",
        "No post-test tuning is permitted from these results.",
    ],
}

(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")

print("OUTPUT:", OUT)
print("STATUS: FROZEN_SHARP_CYCLE25_EVALUATION_COMPLETE_NO_REFITTING")


## 6. Interpretation guardrail

The Cycle-25 results in this notebook are independent-test results for the frozen SHARP pipelines.

They may be **reported**, but they must not be used to modify model weights, calibration, thresholds, features, or test-sample rules.

Any later model change defines a new model and requires a new, untouched evaluation design.
